# SI codes

Plotting code for SI panels. The code reads the minimal result tables in `data/` and keeps the original plotting styles used in the source notebooks.

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

DATA = Path("data")
FIG = Path("figures")
FIG.mkdir(exist_ok=True)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.family'] = 'Arial'
colormap = plt.cm.coolwarm
values = np.linspace(0, 1, 5)
colors = sns.color_palette("vlag", 5)


In [ ]:
# SI from fig1.ipynb
plt.rcParams['font.size'] = 25
speed = pd.read_csv(DATA / "si_fig1_polarization_speed.csv")
for topic, out in [('Partisan Alignment', 'politics_speed.pdf'), ('Gun Control', 'guncontrol_speed.pdf')]:
    all_df = speed.loc[speed['topic'] == topic]
    plt.figure(figsize=(6,5))
    sns.lineplot(all_df, x='time', y='bias_diff', hue='condition', palette=sns.color_palette("ch:s=.25,rot=-.25", 3), style='condition', lw=3, errorbar=None)
    plt.legend()
    plt.xlabel('Time')
    plt.ylabel('Speed of Polarization')
    plt.title(topic)
    plt.savefig(FIG / out, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

short_colors = [colors[0], colors[2], colors[4]]
for fname, outname in [('si_fig1_opinion_bars_simulation.csv', 'opinion_bars_simulation.pdf'), ('si_fig1_opinion_bars_empirical.csv', 'opinion_bars_empirical.pdf')]:
    df = pd.read_csv(DATA / fname)
    title_list = df['topic'].drop_duplicates().tolist()
    fig, axs = plt.subplots(1, 3, figsize=(12,4))
    for i, title in enumerate(title_list):
        sub = df.loc[df['topic'] == title].set_index('group').loc[['Left Leaning','Neutral','Right Leaning']]
        vals = sub['proportion'].to_numpy()
        axs[i].bar(np.arange(3), vals, color=short_colors, edgecolor='k')
        for x, val in enumerate(vals):
            axs[i].text(x-0.3, val, '{:.2f}%'.format(val*100), fontsize=15)
        axs[i].set_title(title, fontsize=20, fontweight='bold')
        axs[i].set_xticks(np.arange(3))
        axs[i].set_xticklabels(['Left\nLeaning','Neutral','Right\nLeaning'], fontsize=20)
        axs[i].set_ylim(0, np.max(vals)*1.2)
    plt.tight_layout()
    plt.savefig(FIG / outname, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()


In [ ]:
# SI from fig2.ipynb
plt.rcParams['font.size'] = 20
pair = pd.read_csv(DATA / "si_fig2_pairwise_opinion_results.csv")
for topic_key, suptitle, out in [('AbortionBan','Abortion Ban','SI_AbortionBan_pairwise.pdf'),('Politics','Partisan Alignment','SI_Politics_pairwise.pdf'),('GunControl','Gun Control','SI_GunControl_pairwise.pdf'),('Obamacare','Obamacare','SI_Obamacare_pairwise.pdf'),('EducationReform','Education Reform','SI_EducationReform_pairwise.pdf')]:
    sub = pair.loc[pair['topic_key'] == topic_key]
    fig, axs = plt.subplots(2, 5, figsize=(17, 6.5))
    for row, condition in enumerate(['Original', 'Self-regulated']):
        for j in range(5):
            vals = sub.loc[(sub['condition'] == condition) & (sub['source_group_index'] == j)].sort_values('target_state_index')['proportion'].to_numpy()
            ax = axs[row, j]
            ax.bar(np.arange(len(vals)), vals, color=colors, edgecolor='k', lw=1)
            ax.set_ylim(0,1)
            ax.set_xticks(np.arange(5))
            ax.set_xticklabels('')
    axs[0,0].set_ylabel('Proportion')
    axs[0,2].set_title('Original')
    axs[1,0].set_ylabel('Proportion')
    axs[1,2].set_title('Self-regulated')
    plt.tight_layout()
    plt.subplots_adjust(top=0.85, left=0.01)
    plt.suptitle(suptitle, weight='bold')
    plt.savefig(FIG / out, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

metrics = pd.read_csv(DATA / "si_fig2_error_retry_metrics.csv")
topics_name = ['Obamacare', 'Politics', 'AbortionBan','GunControl','EducationReform']
topics_name_shows = np.array(['Partisan\nAlignment','Obamacare','Abortion\nBan','Gun\nControl','Education\nReform'])
barcolors = ['#AC434E', '#559F59', '#5EA5C5', '#4469A6',  '#7366A7', '#F08743', '#C5B875']
for metric, ylabel, out in [('error_frequency','Frequency of Errors','mistake_frequency.pdf'), ('retry_frequency','Frequency of Retries','retry_frequency.pdf')]:
    plt.figure(figsize=(10,5))
    for i, topic in enumerate(topics_name):
        vals = metrics.loc[(metrics['topic_key'] == topic) & (metrics['metric'] == metric)].set_index('stage')['value']
        plt.bar(i-0.2, vals['initialize'], width=0.2, color=barcolors[i], hatch='\\', edgecolor='w', label='Stage 1')
        plt.bar(i, vals['persuade'], width=0.2, color=barcolors[i], alpha=0.75, hatch='-', edgecolor='w', label='Stage 2')
        plt.bar(i+0.2, vals['final_determine'], width=0.2, color=barcolors[i], alpha=0.5, hatch='/', edgecolor='w', label='Stage 3')
    ax = plt.gca()
    ax.set_xticks(np.arange(5))
    ax.set_xticklabels(topics_name_shows)
    ax.set_ylabel(ylabel)
    plt.savefig(FIG / out, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()


In [ ]:
# SI from fig4.ipynb
plt.rcParams['font.size'] = 25
ts = pd.read_csv(DATA / "si_fig4_control_timeseries.csv")
summary = pd.read_csv(DATA / "si_fig4_control_summary_after_t15.csv")
config = [
    ('Exaggerated misperception', 'Ratio of Agents with\nExaggerated Misperceptions', (1.3,1.7), 'exaggerated_misperception'),
    ('Objective illusion', 'Ratio of Agents with Objective Illusion', (1.3,1.8), 'objective_illusion'),
    ('Stereotyping', 'Ratio of Agents with Stereotyping', (1.3,1.8), 'Stereotyping'),
]
for panel, xlabel, ylim, stem in config:
    sub = ts.loc[ts['panel'] == panel]
    plt.figure(figsize=(12,5))
    sns.lineplot(sub, x='time', y='bias', hue='condition', hue_order=['0%','50%'], style='condition', palette=sns.color_palette("ch:s=.25,rot=-.25", 3), lw=3, errorbar=None)
    plt.legend(loc='lower right')
    plt.xlabel('Time')
    plt.ylabel('Level of Polarization')
    plt.savefig(FIG / f'{stem}_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

    sub2 = summary.loc[summary['panel'] == panel]
    plt.figure(figsize=(6.2,5))
    sns.barplot(sub2, x='condition', y='bias', order=['0%','50%'], capsize=0.2, palette=sns.color_palette("ch:s=.25,rot=-.25", 3))
    plt.ylim(*ylim)
    plt.xlabel(xlabel)
    plt.ylabel('Level of Polarization')
    plt.savefig(FIG / f'{stem}_2.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()


In [ ]:
# SI from intervention_experiments.ipynb
plt.rcParams['font.size'] = 25
barcolors = ['#AC434E', '#F08743', '#559F59', '#5EA5C5', '#4469A6',  '#7366A7', '#F08743']
exp = ['Original','Random Interactions', 'Moderate Opposing Interactions', 'Neutral Elite Signaling', 'No Selective Exposure', 'No Confirmation Bias']

avg = pd.read_csv(DATA / "si_intervention_average_polarization_after_t35.csv")
plt.figure(figsize=(18,7.5))
sns.barplot(avg, y='type', x='polarization', order=exp, orient='h', capsize=0.2, palette=sns.color_palette('tab10'))
plt.xlim(0.8, 1.4)
plt.ylabel('')
plt.xlabel('Average Polarization Level After Intervetions')
plt.savefig(FIG / 'intervetion_average.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()

dist = pd.read_csv(DATA / "si_intervention_opinion_distribution_t40.csv")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(24,16))
for i in range(2):
    for j in range(3):
        ax = axes[i,j]
        sub = dist.loc[dist['type'] == exp[i*3+j]].set_index('state')
        tmp = np.array([sub.loc[s, 'proportion'] if s in sub.index else 0 for s in [-2,-1,0,1,2]])[::-1]
        ax.bar(np.arange(-2,3,1), tmp, color=colors, edgecolor='k')
        for ii in range(5):
            ax.text(ii-2-0.4, tmp[ii]+0.01, '{:.2f}%'.format(tmp[ii]*100))
        ax.set_title(exp[i*3+j])
        ax.set_ylim(0,0.6)
        ax.set_xlabel('Opinion')
        ax.set_xticks([-2,-1,0,1,2])
        ax.set_xticklabels(['Left','Mod.\nLeft','Neutral','Mod.\nRight','Right'])
axes[0,0].set_ylabel('Proportion')
axes[1,0].set_ylabel('Proportion')
plt.tight_layout()
plt.savefig(FIG / 'intervetion_opinions.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()

chg = pd.read_csv(DATA / "si_intervention_change_in_opinion_t34_to_t40.csv")
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(24,16))
for i in range(2):
    for j in range(3):
        ax = axes[i,j]
        sub = chg.loc[chg['type'] == exp[i*3+j]].set_index('delta_abs_opinion')
        tmp = np.array([sub.loc[s, 'proportion'] if s in sub.index else 0 for s in [-2,-1,0,1,2]])
        ax.bar(np.arange(-2,3,1), tmp, color=sns.color_palette('tab10')[i*3+j], edgecolor='k', alpha=0.8)
        ax.text(-2,0.6,'{:.2f}%'.format(np.sum(tmp[:2])*100))
        ax.text(1,0.6,'{:.2f}%'.format(np.sum(tmp[-2:])*100))
        ax.set_title(exp[i*3+j])
        ax.set_ylim(0,1)
        ax.set_xlabel('Change in Opinion')
axes[0,0].set_ylabel('Proportion')
axes[1,0].set_ylabel('Proportion')
plt.tight_layout()
plt.savefig(FIG / 'intervetion_change_in_opinions_orig.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()

net = pd.read_csv(DATA / "si_intervention_change_in_network.csv")
plt.figure(figsize=(18,7.5))
sns.barplot(net, y='type', x='change', orient='h', width=0.5, capsize=0.1, palette=sns.color_palette('tab10'))
plt.ylabel('')
plt.xlabel('Change in Socal Network')
plt.savefig(FIG / 'intervetion_change_in_network.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()


In [ ]:
# SI from NC_experiments/experiments.ipynb
plt.rcParams['font.size'] = 25
temp = pd.read_csv(DATA / "si_nc_temperature_homophily_polarization.csv")
marker_map = {'Temperature=0.5':'o', 'Temperature=1.0':'^', 'Temperature=1.5':'s'}
for y, ylabel, out in [('homophilic_interaction', 'Prop. of Homophilic Int.', 'homointeraction_temperature.pdf'), ('polarization', 'Level of Polarization', 'polarization_temperature.pdf')]:
    fig, ax = plt.subplots(1,1, figsize=(12,5))
    for idx, label in enumerate(['Temperature=0.5','Temperature=1.0','Temperature=1.5']):
        sub = temp.loc[temp['temperature'] == label]
        sns.lineplot(sub, x='timestep', y=y, markers=True, marker=marker_map[label], markersize=10 if label == 'Temperature=1.0' else 8, ax=ax, label=label, errorbar=None, palette=colors[idx])
    ax.set_ylabel(ylabel)
    ax.set_xlabel(r'Time')
    ax.legend(loc='lower right')
    plt.savefig(FIG / out, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

net = pd.read_csv(DATA / "si_nc_temperature_network_change.csv").rename(columns={'temperature':'Exp'})
for y, ylabel, out in [('Node_change','Change Rate in Nodes (%)','change_rate_nodes.pdf'), ('Edge_change','Change Rate in Edges (%)','change_rate_edges.pdf')]:
    plt.figure(figsize=(6,5))
    sns.barplot(net, x='Exp', y=y, hue='Exp', capsize=0.1)
    plt.xlabel('Temperature')
    plt.xticks([0,1,2], [0.5, 1.0, 1.5])
    plt.ylabel(ylabel)
    plt.savefig(FIG / out, bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

flat = pd.read_csv(DATA / "si_nc_flat_earth_opinion_dynamics.csv")
new_colors = ['#717071', '#B4B4B5', '#EEEEEF', '#A3C49E', '#679C5F'][::-1]
plt.figure(figsize=(10,9))
pivot = flat.pivot(index='timestep', columns='state', values='proportion').fillna(0).sort_index(axis=1)
cum = np.cumsum(pivot.to_numpy().T, axis=0)
data_matrix = np.concatenate([np.zeros((1, pivot.shape[0])), cum], axis=0)
for ii in range(1, 6):
    plt.fill_between(pivot.index, data_matrix[ii-1,:], data_matrix[ii,:], color=new_colors[ii-1])
plt.ylim(0,1)
plt.xlim(0, pivot.index.max())
plt.xticks([j for j in range(0, pivot.index.max()+1, 5)])
plt.xlabel(r'Time')
plt.ylabel('Proportion')
plt.savefig(FIG / 'flat_earth_1.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()

openm = pd.read_csv(DATA / "si_nc_openminded_interventions.csv")
fig, ax = plt.subplots(1,1, figsize=(10,4))
sns.barplot(openm, y='type', x='polarization', order=['Original', 'Opposing Interactions','Openmindedness', 'Openmindedness+Opposing Interactions'], capsize=0.2, palette=barcolors, saturation=1, orient='h')
plt.yticks([0,1,2,3], ['Orig', 'Oppose', 'Open', 'Open+Oppose'])
plt.xlim(1.4, 2.0)
plt.xticks([1.4, 1.6, 1.8, 2.0])
plt.xlabel(r'Polarization Level after Intervention')
plt.ylabel('')
plt.savefig(FIG / 'openminded_experiments.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()


In [ ]:
# SI from argument_extraction.ipynb
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 20
arg = pd.read_csv(DATA / "si_argument_toward_pro_counts.csv")
TOPICS_PLOT = ['Politics', 'Gun_Control', 'Abortion']
TOPIC_DISPLAY = {'Abortion': 'Abortion Ban', 'Gun_Control': 'Gun Control', 'Politics': 'Partisan Alignment'}
x = np.arange(len(TOPICS_PLOT))
w = 0.35
means_prev = [arg.loc[arg.topic==t, 'toward_pro_previous'].mean() for t in TOPICS_PLOT]
means_curr = [arg.loc[arg.topic==t, 'toward_pro_current'].mean() for t in TOPICS_PLOT]
sems_prev = [stats.sem(arg.loc[arg.topic==t, 'toward_pro_previous']) for t in TOPICS_PLOT]
sems_curr = [stats.sem(arg.loc[arg.topic==t, 'toward_pro_current']) for t in TOPICS_PLOT]
fig, ax = plt.subplots(1, 1, figsize=(3.5, 3))
ax.set_facecolor('white')
fig.patch.set_facecolor('white')
ax.bar(x - w/2, means_prev, w, label='Before polarizing', color='#4A90A4', yerr=sems_prev, capsize=2.5, error_kw={'linewidth': 1})
ax.bar(x + w/2, means_curr, w, label='After polarizing', color='#C75B4F', yerr=sems_curr, capsize=2.5, error_kw={'linewidth': 1})
ax.set_xticks(x)
ax.set_xticklabels([TOPIC_DISPLAY[t] for t in TOPICS_PLOT])
ax.set_ylabel('# of supporting arguments', fontsize=8)
ax.set_xlabel('Topic', fontsize=8)
ax.legend(loc='upper right', fontsize=7)
ax.tick_params(axis='both', labelsize=7)
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
for i, topic in enumerate(TOPICS_PLOT):
    prev_arr = arg.loc[arg.topic==topic, 'toward_pro_previous'].to_numpy()
    curr_arr = arg.loc[arg.topic==topic, 'toward_pro_current'].to_numpy()
    _, p = stats.ttest_rel(curr_arr, prev_arr)
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ax.text(i, max(means_prev[i], means_curr[i]) + max(sems_prev[i], sems_curr[i]) + 0.2, sig, ha='center', fontsize=7)
y_top = max(max(means_prev[i] + sems_prev[i] for i in range(len(TOPICS_PLOT))), max(means_curr[i] + sems_curr[i] for i in range(len(TOPICS_PLOT)))) + 0.9
ax.set_ylim(0, y_top)
plt.tight_layout()
plt.savefig(FIG / 'argument_toward_pro_before_after_by_topic.pdf', bbox_inches='tight', dpi=300)
plt.show()

pop = pd.read_csv(DATA / "si_argument_popularity_top10.csv")
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
fig.patch.set_facecolor('white')
for idx, topic in enumerate(TOPICS_PLOT):
    ax = axes[idx]
    ax.set_facecolor('white')
    sub = pop.loc[pop['topic'] == topic].sort_values('rank')
    labels = sub['argument_text'].tolist()
    values = sub['relative_frequency'].tolist()
    y_pos = np.arange(len(labels))[::-1]
    ax.barh(y_pos, values, color='#bfbfbf', height=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('Relative frequency', fontsize=7)
    ax.set_title(TOPIC_DISPLAY[topic], fontsize=8)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.tick_params(axis='both', labelsize=7)
plt.tight_layout()
plt.savefig(FIG / 'argument_popularity_top10_by_topic.pdf', bbox_inches='tight', dpi=300)
plt.show()


In [ ]:
# SI from affection.ipynb
plt.rcParams['font.size'] = 20
rating = pd.read_csv(DATA / "si_affection_impression_ratings.csv")
x_order = ['positive_to_positive','negative_to_negative', 'neutral_to_neutral', 'positive_to_negative', 'negative_to_positive', 'neutral_to_positive', 'neutral_to_negative', 'positive_to_neutral', 'negative_to_neutral']
y_labels = [r'left -> left', r'right -> right', r'neutral -> neutral', r'left -> right', r'right -> left', r'neutral -> left', r'neutral -> right', r'left -> neutral', r'right -> neutral']
bar_palette = ['#5B9DBC']*3 + ['#DC7887']*2 + ['#ADADAD']*4
for ee in sorted(rating['epoch'].unique()):
    plt.figure()
    sns.barplot(rating.loc[rating['epoch'] == ee], y='type', x='score', order=x_order, orient='h', palette=bar_palette, capsize=0.2)
    plt.yticks(np.arange(9), y_labels)
    plt.xlabel('Impression Ratings')
    plt.ylabel('')
    plt.xlim(1,5)
    plt.title('Epoch {}'.format(ee))
    plt.savefig(FIG / './epoch_{}.pdf'.format(ee), bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

top = pd.read_csv(DATA / "si_affection_top_descriptions.csv")
for x_idx, xx in enumerate(x_order):
    sub = top.loc[top['type'] == xx].sort_values('frequency', ascending=False).head(10)
    plt.figure()
    bar_colors = []
    for pp in sub['property']:
        if pp == 'positive':
            bar_colors.append('#01B468')
        elif pp == 'negative':
            bar_colors.append('#FF0000')
        else:
            bar_colors.append('#ADADAD')
    plt.barh(np.arange(len(sub))[::-1], sub['fraction_of_agents'].to_numpy(), color=bar_colors)
    plt.yticks(np.arange(len(sub))[::-1], sub['word'].to_numpy())
    plt.title(y_labels[x_idx])
    plt.xlim(0,1)
    plt.xlabel('Fraction of Agents')
    plt.savefig(FIG / './descb_{}.pdf'.format(xx), bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()

sent = pd.read_csv(DATA / "si_affection_description_sentiment_fractions.csv")
for x_idx, xx in enumerate(x_order):
    sub = sent.loc[sent['type'] == xx]
    vals = {'positive':0, 'negative':0, 'neutral':0}
    for _, row in sub.iterrows():
        vals[row['property']] = row['fraction_of_descriptions']
    plt.figure()
    plt.bar(np.arange(3), [vals['positive'], vals['negative'], vals['neutral']], color=['#01B468', '#FF0000', '#ADADAD'])
    plt.title(y_labels[x_idx])
    plt.ylabel('Fraction of Descriptions')
    plt.xticks([0,1,2], ['Positive', 'Negative', 'Neutral'])
    plt.xlabel('Sentiment')
    plt.savefig(FIG / './overall_descb_{}.pdf'.format(xx), bbox_inches='tight', pad_inches=0.1, dpi=300)
    plt.show()
